<a href="https://colab.research.google.com/github/harisharma800/Spam_detection_using_Pytorch/blob/main/spam_detection_using_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
from nltk.corpus import stopwords
import nltk
import string
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split

In [4]:
df=pd.read_csv("/content/spam.csv", encoding="ISO-8859-1")

In [5]:
df=df.rename(columns={'v2':'message'})

In [6]:
df=df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])

In [7]:
df['v1']=df['v1'].replace({'ham':0, 'spam':1})

/tmp/ipykernel_10051/788310506.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['v1']=df['v1'].replace({'ham':0, 'spam':1})


In [8]:
df.describe()

,v1
count,5572.000000
mean,0.134063
std,0.340751
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   v1       5572 non-null   int64 
 1   message  5572 non-null   object
dtypes: int64(1), object(1)
memory usage: 87.2+ KB


In [10]:
df.head()

,v1,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
lowerCase=lambda x: x.lower()
df['message']=df['message'].apply(lowerCase)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [12]:
df['message']=df['message'].str.replace('[^\w\s]','')

<>:1: SyntaxWarning: invalid escape sequence '\w'
<>:1: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_10051/764003539.py:1: SyntaxWarning: invalid escape sequence '\w'
  df['message']=df['message'].str.replace('[^\w\s]','')


In [13]:
tokenizer=lambda x : word_tokenize(x)
df['message']=df['message'].apply(tokenizer)

In [14]:
stop_words=lambda x : [word for word in x if word not in stopwords.words('english')]
df['message']=df['message'].apply(stop_words)

In [15]:
labels=torch.tensor(df['v1'],dtype=torch.float32)

In [16]:
vectorizer=CountVectorizer()

In [17]:
df['message']=df['message'].apply(lambda x: ' '.join(x))

In [18]:
X=vectorizer.fit_transform(df['message']).toarray()

In [19]:
X=torch.tensor(X,  dtype=torch.float32)

In [20]:
df['message'].shape

(5572,)

In [21]:
print('voca:',vectorizer.get_feature_names_out())
print(X)

voca: ['00' '000' '000pes' ... 'ûïharry' 'ûò' 'ûówell']
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])


In [22]:
X.shape[1]

8606

In [23]:
labels.shape

torch.Size([5572])

In [24]:
model=nn.Sequential(
    nn.Linear(X.shape[1],8),
    nn.ReLU(),
    nn.Linear(8,1)
)

In [25]:
print(X[:5])
print(X.shape)

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
torch.Size([5572, 8606])


In [26]:
loss_fn=nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(model.parameters(), lr=0.01)

In [27]:
model(X).squeeze(1).shape

torch.Size([5572])

In [36]:
for epcho in range(1000):
  predictions=model(X).squeeze(1)
  loss=loss_fn(predictions,labels)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if (epcho+1)%100==0:
    print(f'epcho{epcho+1},loss:{loss.item():.4f}')

epcho100,loss:0.0002
epcho200,loss:0.0002
epcho300,loss:0.0002
epcho400,loss:0.0002
epcho500,loss:0.0002
epcho600,loss:0.0002
epcho700,loss:0.0002
epcho800,loss:0.0002
epcho900,loss:0.0002
epcho1000,loss:0.0002


In [41]:
new_text=["you won 100 rupee"]
X_new=vectorizer.transform(new_text).toarray()
X_new=torch.tensor(X_new, dtype=torch.float32)
with torch.no_grad():
    probs = torch.sigmoid(model(X_new).squeeze(1))
    preds = (probs >= 0.5).int()
print("Spam probability:", probs.item())
print("Prediction:", "Spam" if preds.item() == 1 else "Not Spam")

Spam probability: 0.05956925451755524
Prediction: Not Spam
